# Fine-tuning do classificador de e-mails

Este notebook:

1. lê as classificações humanas de `ground_truth_emails.xlsx`;
2. reúne os JSON em `research/extracted_emails` e `data/extracted_emails`;
3. cria uma divisão **estratificada 80/20**, preservando aproximadamente a proporção das três classes;
4. faz fine-tuning do XLM-RoBERTa com os 80% de treino;
5. avalia uma única vez nos 20% de teste;
6. disponibiliza uma célula final para experimentar e-mails novos.

O conjunto de teste não é usado durante o treino. A semente fixa torna a divisão reproduzível.

In [1]:
import json
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from openpyxl import load_workbook
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)


def find_project_root():
    # Permite executar o notebook a partir da raiz ou da pasta tests.
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "research/ground_truth/ground_truth_emails.xlsx").exists():
            return candidate
    raise FileNotFoundError("Não encontrei a raiz do projeto GlobalBrico.")


ROOT = find_project_root()
EXCEL_PATH = ROOT / "research/ground_truth/ground_truth_emails.xlsx"
EMAIL_DIRS = [ROOT / "research/extracted_emails", ROOT / "data/extracted_emails"]
BASE_MODEL = "joeddav/xlm-roberta-large-xnli"
MODEL_OUTPUT = ROOT / "src/models/xlm_roberta_large_email_512"

LABELS = ("Pedido de Informação", "Pedido de Encomenda", "SPAM")
SEED = 42
TEST_SIZE = 0.20
MAX_LENGTH = 512
EPOCHS = 5

np.random.seed(SEED)
torch.manual_seed(SEED)
print("Raiz do projeto:", ROOT)
print("Dispositivo disponível:", "CUDA" if torch.cuda.is_available() else "CPU/MPS")

/Users/alexandre.sousa.ftp/GlobalBrico/.GB/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: Não encontrei a raiz do projeto GlobalBrico.

## Carregar os dados rotulados

Os JSON antigos e os exemplos adicionados manualmente estão em duas pastas. Quando um UID aparece nas duas, o conteúdo tem de ser igual; assim evitamos substituir silenciosamente um exemplo por outro.

In [ ]:
def normalize_email(email):
    # Retém apenas os campos usados pelo classificador para comparar duplicados.
    return {
        key: email.get(key) or ""
        for key in ("subject", "from", "text", "html")
    }


def conversation_key(subject):
    # Mantém respostas/forwards da mesma conversa no mesmo conjunto.
    subject = re.sub(
        r"^\s*(?:\*+SPAM\*+|\[SPAM\]|SPAM)[\s:_-]*", "", subject or "", flags=re.I
    )
    while True:
        cleaned = re.sub(r"^\s*(?:re|fw|fwd|enc)\s*:\s*", "", subject, flags=re.I)
        if cleaned == subject:
            break
        subject = cleaned
    subject = unicodedata.normalize("NFKD", subject).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", subject.casefold()).strip()


def email_text(email):
    # Mesmo pré-processamento usado na classificação de produção.
    subject = re.sub(
        r"^\s*(?:\*+SPAM\*+|\[SPAM\]|SPAM\b)[\s:_-]*",
        "",
        email.get("subject") or "",
        flags=re.I,
    )
    return "\n".join(
        part
        for part in (
            f"Assunto: {subject}",
            f"Remetente: {email.get('from') or ''}",
            email.get("text") or email.get("html") or "",
        )
        if part.strip()
    )


def load_human_labels(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = workbook["Revisão"].iter_rows(values_only=True)
        header = next(row for row in rows if "UID" in row and "Label correta" in row)
        uid_col = header.index("UID")
        label_col = header.index("Label correta")
        labels = {}
        for row in rows:
            uid = str(row[uid_col] or "").removesuffix(".0")
            label = str(row[label_col] or "").strip()
            if not uid or not label:
                continue
            if label not in LABELS:
                raise ValueError(f"Label inválida para {uid}: {label}")
            if uid in labels:
                raise ValueError(f"UID repetido no Excel: {uid}")
            labels[uid] = label
        return labels
    finally:
        workbook.close()


def load_emails(directories):
    emails = {}
    origins = {}
    for directory in directories:
        for path in sorted(directory.glob("*.json")):
            if path.name == "summary.json":
                continue
            email = json.loads(path.read_text(encoding="utf-8"))
            uid = str(email.get("uid") or "")
            if not uid:
                raise ValueError(f"UID ausente: {path}")
            if uid in emails and normalize_email(emails[uid]) != normalize_email(email):
                raise ValueError(f"Conteúdo diferente para o UID duplicado {uid}: {origins[uid]} e {path}")
            emails[uid] = email
            origins[uid] = path
    return emails, origins


human_labels = load_human_labels(EXCEL_PATH)
emails, origins = load_emails(EMAIL_DIRS)
missing = sorted(set(human_labels) - set(emails))
if missing:
    raise ValueError(f"Faltam JSON para os UIDs: {missing}")

records = [
    {
        "uid": uid,
        "label": label,
        "text": email_text(emails[uid]),
        "source": str(origins[uid].relative_to(ROOT)),
        "conversation": conversation_key(emails[uid].get("subject") or ""),
    }
    for uid, label in human_labels.items()
]
data = pd.DataFrame(records)

print(f"Exemplos rotulados: {len(data)}")
display(data["label"].value_counts().rename("Total").to_frame())
data.head(3)

Exemplos rotulados: 51


,Total
label,
Pedido de Informação,25
SPAM,15
Pedido de Encomenda,11


,uid,label,text,source,conversation
0,27391,Pedido de Informação,Assunto: pedido de orçamento\nRemetente: jasil...,research/extracted_emails/27391_pedido_de_oram...,pedido de orcamento
1,27390,Pedido de Informação,Assunto: Pedido de orçamento - Obras\nRemetent...,research/extracted_emails/27390_Pedido_de_oram...,pedido de orcamento obras
2,27182,Pedido de Informação,Assunto: Solicitação de Cotação e Especificaçõ...,data/extracted_emails/27182_Solicitao_de_Cotao...,solicitacao de cotacao e especificacoes do ped...


## Divisão estratificada 80/20

A divisão usa cinco blocos estratificados e escolhe como teste o bloco mais próximo de 20% e da distribuição global das classes. Respostas e forwards com o mesmo assunto normalizado ficam sempre no mesmo conjunto, reduzindo fuga de informação entre treino e teste. Com 51 exemplos, 80/20 só pode ser aproximado.

In [ ]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
candidates = []
target_counts = data["label"].value_counts() * TEST_SIZE
for train_indices, test_indices in splitter.split(
    data, y=data["label"], groups=data["conversation"]
):
    counts = data.iloc[test_indices]["label"].value_counts()
    distance = abs(len(test_indices) - len(data) * TEST_SIZE)
    distance = sum(abs(counts.get(label, 0) - target_counts.get(label, 0)) for label in LABELS)
    candidates.append((distance, train_indices, test_indices))

_, train_indices, test_indices = min(candidates, key=lambda candidate: candidate[0])
train_data = data.iloc[train_indices].reset_index(drop=True)
test_data = data.iloc[test_indices].reset_index(drop=True)

distribution = pd.concat(
    {
        "Total": data["label"].value_counts(),
        "Treino": train_data["label"].value_counts(),
        "Teste": test_data["label"].value_counts(),
    },
    axis=1,
).reindex(LABELS).fillna(0).astype(int)
distribution["Treino %"] = (distribution["Treino"] / len(train_data) * 100).round(1)
distribution["Teste %"] = (distribution["Teste"] / len(test_data) * 100).round(1)

assert set(train_data["uid"]).isdisjoint(test_data["uid"])
assert len(train_data) + len(test_data) == len(data)
display(distribution)
print(f"Treino: {len(train_data)} | Teste: {len(test_data)}")

,Total,Treino,Teste,Treino %,Teste %
label,,,,,
Pedido de Informação,25,20,5,48.8,50.0
Pedido de Encomenda,11,9,2,22.0,20.0
SPAM,15,12,3,29.3,30.0


Treino: 41 | Teste: 10


## Tokenização e fine-tuning

O modelo aceita até 512 tokens. O `DataCollatorWithPadding` ajusta o preenchimento ao maior e-mail de cada lote, evitando preencher todos os exemplos até 512 tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


class EmailDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length):
        self.items = []
        for row in frame.itertuples(index=False):
            item = tokenizer(row.text, truncation=True, max_length=max_length)
            item["labels"] = LABELS.index(row.label)
            self.items.append(item)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


train_dataset = EmailDataset(train_data, tokenizer, MAX_LENGTH)
test_dataset = EmailDataset(test_data, tokenizer, MAX_LENGTH)

token_lengths = data["text"].map(
    lambda text: len(tokenizer(text, truncation=False, add_special_tokens=True)["input_ids"])
)
print(f"Mediana: {token_lengths.median():.0f} tokens")
print(f"Acima de {MAX_LENGTH}: {(token_lengths > MAX_LENGTH).sum()} de {len(data)} e-mails")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (794 > 512). Running this sequence through the model will result in indexing errors


Mediana: 333 tokens
Acima de 512: 16 de 51 e-mails


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)),
    label2id={label: index for index, label in enumerate(LABELS)},
)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
    generator=generator,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, round(total_steps * 0.10)),
    num_training_steps=total_steps,
)

print(f"Fine-tuning em {device}: {EPOCHS} épocas, {total_steps} passos")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    progress = tqdm(
        train_loader,
        desc=f"Época {epoch + 1}/{EPOCHS}",
        unit="lote",
        leave=True,
    )
    for step, batch in enumerate(progress, start=1):
        batch = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        output = model(**batch)
        output.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += output.loss.item()
        progress.set_postfix(
            loss=f"{output.loss.item():.4f}",
            loss_media=f"{total_loss / step:.4f}",
        )
    print(f"Época {epoch + 1}/{EPOCHS} concluída — loss média: {total_loss / len(train_loader):.4f}")

MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(MODEL_OUTPUT))
tokenizer.save_pretrained(str(MODEL_OUTPUT))
print("Modelo guardado em:", MODEL_OUTPUT)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5640.03it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fine-tuning em cuda: 5 épocas, 105 passos


Época 1/5: 100%|██████████| 21/21 [00:09<00:00,  2.28lote/s, loss=1.3404, loss_media=1.6279]


Época 1/5 concluída — loss média: 1.6279


Época 2/5: 100%|██████████| 21/21 [00:08<00:00,  2.58lote/s, loss=0.3239, loss_media=1.0300]


...

Época 4/5 concluída — loss média: 0.6996


Época 5/5: 100%|██████████| 21/21 [00:08<00:00,  2.52lote/s, loss=0.0004, loss_media=0.4278]


Época 5/5 concluída — loss média: 0.4278


Writing model shards: 100%|██████████| 1/1 [00:15<00:00, 15.56s/it]

Modelo guardado em: /home/alex/Desktop/classification_globalbrico/src/models/xlm_roberta_large_email_512


## Avaliação no conjunto de teste

Com apenas 11 exemplos no teste, as métricas variam bastante. A matriz de confusão e as previsões individuais ajudam a perceber que tipos de erro ocorreram.

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collator,
)

model.eval()
all_logits = []
expected_ids = []
with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop("labels")
        inputs = {key: value.to(device) for key, value in batch.items()}
        all_logits.append(model(**inputs).logits.cpu())
        expected_ids.extend(labels.tolist())

logits = torch.cat(all_logits).numpy()
expected_ids = np.asarray(expected_ids)
predicted_ids = np.argmax(logits, axis=-1)

print(classification_report(
    expected_ids,
    predicted_ids,
    labels=range(len(LABELS)),
    target_names=LABELS,
    zero_division=0,
))

confusion = pd.DataFrame(
    confusion_matrix(expected_ids, predicted_ids, labels=range(len(LABELS))),
    index=pd.Index(LABELS, name="Label correta"),
    columns=pd.Index(LABELS, name="Previsão"),
)
display(confusion)

test_results = test_data[["uid", "label", "source"]].copy()
test_results["previsão"] = [LABELS[index] for index in predicted_ids]
test_results["correto"] = test_results["label"] == test_results["previsão"]
display(test_results.sort_values(["correto", "label", "uid"]))

                      precision    recall  f1-score   support

Pedido de Informação       0.75      0.60      0.67         5
 Pedido de Encomenda       0.50      0.50      0.50         2
                SPAM       0.75      1.00      0.86         3

            accuracy                           0.70        10
           macro avg       0.67      0.70      0.67        10
        weighted avg       0.70      0.70      0.69        10



Previsão,Pedido de Informação,Pedido de Encomenda,SPAM
Label correta,,,
Pedido de Informação,3,1,1
Pedido de Encomenda,1,1,0
SPAM,0,0,3


,uid,label,source,previsão,correto
9,ENC-2026-0015,Pedido de Encomenda,data/extracted_emails/ENC-2026-0015.json,Pedido de Informação,False
6,ENC-2026-0009,Pedido de Informação,data/extracted_emails/ENC-2026-0009.json,SPAM,False
3,ENC-2026-0022,Pedido de Informação,data/extracted_emails/ENC-2026-0022.json,Pedido de Encomenda,False
1,ENC-2026-0017,Pedido de Encomenda,data/extracted_emails/ENC-2026-0017.json,Pedido de Encomenda,True
0,21288,Pedido de Informação,data/extracted_emails/21288_Pedido_de_Cotao_-R...,Pedido de Informação,True
8,ENC-2026-0014,Pedido de Informação,data/extracted_emails/ENC-2026-0014.json,Pedido de Informação,True
4,ENC-2026-0024,Pedido de Informação,data/extracted_emails/ENC-2026-0024.json,Pedido de Informação,True
7,ENC-2026-0011,SPAM,data/extracted_emails/ENC-2026-0011.json,SPAM,True
2,ENC-2026-002,SPAM,data/extracted_emails/ENC-2026-002.json,SPAM,True
5,ENC-2026-0026,SPAM,data/extracted_emails/ENC-2026-0026.json,SPAM,True


## Testar e-mails novos

Edita apenas `NOVOS_EMAILS` e volta a executar esta célula. Estes e-mails não entram no treino nem alteram o modelo.

In [ ]:
NOVOS_EMAILS = [
    {
  "uid": "27450",
  "message_id": "CAM\u002BVw5jzhNcSKhfCXH1oAhdO0TgcjZtuHj4ZZEWXccZJvr4bOA@mail.gmail.com",
  "in_reply_to": "CAGNSXXztBQKSX6SSO6G\u002BgDYS4Xcizu17rq=ct5BWQ3XZ2GeYYQ@mail.gmail.com",
  "references": [
    "CAGNSXXztBQKSX6SSO6G\u002BgDYS4Xcizu17rq=ct5BWQ3XZ2GeYYQ@mail.gmail.com"
  ],
  "subject": "Fwd: Pedido de or\u00E7amento formal para materiais, ferramentas e equipamentos \u2013 Rodrigo \u0026 Soares, Lda.",
  "from": "\u0022GLOBALBRICO Apoio ao Cliente\u0022 \u003Capoioaocliente@globalbrico.pt\u003E",
  "to": "\u0022GLOBALBRICO ENCOMENDAS\u0022 \u003Cencomendas@globalbrico.pt\u003E",
  "date": "Thu, 17 Sep 2026 12:34:13 \u002B0100",
  "text": "---------- Forwarded message ---------\nDe: Rodrigo Soares \u003Crodrigoesoares.geral@gmail.com\u003E\nDate: quinta, 17/09/2026 \u00E0(s) 11:44\nSubject: Pedido de or\u00E7amento formal para materiais, ferramentas e\nequipamentos \u2013 Rodrigo \u0026 Soares, Lda.\nTo: \u003Capoioaocliente@globalbrico.pt\u003E\n\n\nExmos. Senhores,\n\nNo \u00E2mbito da candidatura aos apoios de recupera\u00E7\u00E3o de preju\u00EDzos causados\npor inc\u00EAndios na nossa explora\u00E7\u00E3o agr\u00EDcola, vimos por este meio solicitar a\nV. Exas. o envio de um or\u00E7amento formal para os seguintes artigos\ndispon\u00EDveis no vosso cat\u00E1logo:\n\n   -\n\n   *01 Compressor de ar* \u2013 Qtd: 1\n   -\n\n   *01 Rebarbadora* \u2013 Qtd: 1\n   -\n\n   *01 Extens\u00E3o em rolo com enrolador (50m)* \u2013 Qtd: 1\n   -\n\n   *02 P\u00E1s* de agricultura \u2013 Qtd: 2\n   -\n\n   *02 Extintores F27A 6Kg* \u2013 Qtd: 2\n   -\n\n   *06 Prateleiras de arruma\u00E7\u00E3o met\u00E1licas* \u2013 Qtd: 6\n   -\n\n   *01 Mesa de apoio de trabalho* \u2013 Qtd: 1\n   -\n\n   *01 Desenrolador de arame* \u2013 Qtd: 1\n   -\n\n   *Tintas e diluentes* para manuten\u00E7\u00E3o \u2013 Qtd: 1 conjunto\n\nAgradecemos que a proposta comercial discrimine explicitamente:\n\n   -\n\n   Os valores unit\u00E1rios e totais de cada artigo;\n   -\n\n   O valor do IVA aplic\u00E1vel;\n   -\n\n   Os custos de transporte/portes de entrega (ou indica\u00E7\u00E3o para\n   levantamento);\n   -\n\n   O prazo de entrega estimado e a validade da proposta (requisito\n   obrigat\u00F3rio para o processo de candidatura).\n\nCertos da vossa melhor aten\u00E7\u00E3o, aguardamos o envio do documento formal.\n\nCom os melhores cumprimentos,\n\n*Altamiro Soares*\n\nRodrigo \u0026 Soares, Lda.\n\nNIF: 510923135\n\nTelem\u00F3vel: 968 080 690\n\n\n--\n",
  "html": "\u003Cdiv dir=\u0022ltr\u0022\u003E\u003Cbr\u003E\u003Cbr\u003E\u003Cdiv class=\u0022gmail_quote gmail_quote_container\u0022\u003E\u003Cdiv dir=\u0022ltr\u0022 class=\u0022gmail_attr\u0022\u003E---------- Forwarded message ---------\u003Cbr\u003EDe: \u003Cstrong class=\u0022gmail_sendername\u0022 dir=\u0022auto\u0022\u003ERodrigo Soares\u003C/strong\u003E \u003Cspan dir=\u0022auto\u0022\u003E\u0026lt;\u003Ca href=\u0022mailto:rodrigoesoares.geral@gmail.com\u0022\u003Erodrigoesoares.geral@gmail.com\u003C/a\u003E\u0026gt;\u003C/span\u003E\u003Cbr\u003EDate: quinta, 17/09/2026 \u00E0(s) 11:44\u003Cbr\u003ESubject: Pedido de or\u00E7amento formal para materiais, ferramentas e equipamentos \u2013 Rodrigo \u0026amp; Soares, Lda.\u003Cbr\u003ETo:  \u0026lt;\u003Ca href=\u0022mailto:apoioaocliente@globalbrico.pt\u0022\u003Eapoioaocliente@globalbrico.pt\u003C/a\u003E\u0026gt;\u003Cbr\u003E\u003C/div\u003E\u003Cbr\u003E\u003Cbr\u003E\u003Cdiv dir=\u0022ltr\u0022\u003E\u003Cp\u003EExmos. Senhores,\u003C/p\u003E\u003Cp\u003ENo \u00E2mbito da candidatura aos apoios de recupera\u00E7\u00E3o de preju\u00EDzos causados por inc\u00EAndios na nossa explora\u00E7\u00E3o agr\u00EDcola, vimos por este meio solicitar a V. Exas. o envio de um or\u00E7amento formal para os seguintes artigos dispon\u00EDveis no vosso cat\u00E1logo:\u003C/p\u003E\u003Cul\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E01 Compressor de ar\u003C/b\u003E \u2013 Qtd: 1\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E01 Rebarbadora\u003C/b\u003E \u2013 Qtd: 1\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E01 Extens\u00E3o em rolo com enrolador (50m)\u003C/b\u003E \u2013 Qtd: 1\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E02 P\u00E1s\u003C/b\u003E de agricultura \u2013 Qtd: 2\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E02 Extintores F27A 6Kg\u003C/b\u003E \u2013 Qtd: 2\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E06 Prateleiras de arruma\u00E7\u00E3o met\u00E1licas\u003C/b\u003E \u2013 Qtd: 6\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E01 Mesa de apoio de trabalho\u003C/b\u003E \u2013 Qtd: 1\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003E01 Desenrolador de arame\u003C/b\u003E \u2013 Qtd: 1\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003E\u003Cb\u003ETintas e diluentes\u003C/b\u003E para manuten\u00E7\u00E3o \u2013 Qtd: 1 conjunto\u003C/p\u003E\u003C/li\u003E\u003C/ul\u003E\u003Cp\u003EAgradecemos que a proposta comercial discrimine explicitamente:\u003C/p\u003E\u003Cul\u003E\u003Cli\u003E\u003Cp\u003EOs valores unit\u00E1rios e totais de cada artigo;\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003EO valor do IVA aplic\u00E1vel;\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003EOs custos de transporte/portes de entrega (ou indica\u00E7\u00E3o para levantamento);\u003C/p\u003E\u003C/li\u003E\u003Cli\u003E\u003Cp\u003EO prazo de entrega estimado e a validade da proposta (requisito obrigat\u00F3rio para o processo de candidatura).\u003C/p\u003E\u003C/li\u003E\u003C/ul\u003E\u003Cp\u003ECertos da vossa melhor aten\u00E7\u00E3o, aguardamos o envio do documento formal.\u003C/p\u003E\u003Cp\u003ECom os melhores cumprimentos,\u003C/p\u003E\u003Cp\u003E\u003Cb\u003EAltamiro Soares\u003C/b\u003E\u003C/p\u003E\u003Cp\u003ERodrigo \u0026amp; Soares, Lda.\u003C/p\u003E\u003Cp\u003ENIF: 510923135\u003C/p\u003E\u003Cp\u003ETelem\u00F3vel: 968 080 690\u003C/p\u003E\u003C/div\u003E\n\u003C/div\u003E\u003Cdiv\u003E\u003Cbr clear=\u0022all\u0022\u003E\u003C/div\u003E\u003Cdiv\u003E\u003Cbr\u003E\u003C/div\u003E\u003Cspan class=\u0022gmail_signature_prefix\u0022\u003E-- \u003C/span\u003E\u003Cbr\u003E\u003Cdiv dir=\u0022ltr\u0022 class=\u0022gmail_signature\u0022 data-smartmail=\u0022gmail_signature\u0022\u003E\u003Cdiv dir=\u0022ltr\u0022\u003E\u003Cimg width=\u0022420\u0022 height=\u002287\u0022 src=\u0022https://ci3.googleusercontent.com/mail-sig/AIorK4zUEibtFWdubi9MeYbFa5BncSGAYw27pP-bEqkIjMDyCi-PDS1J6bJnPzLbxaxBCdlJGEYhvJSbZcZd\u0022\u003E\u003Cbr\u003E\u003C/div\u003E\u003C/div\u003E\u003C/div\u003E\n",
  "folder": "INBOX"
}
]

if not NOVOS_EMAILS:
    print("Adiciona um ou mais e-mails à lista NOVOS_EMAILS.")
else:
    model.eval()
    rows = []
    for number, email in enumerate(NOVOS_EMAILS, start=1):
        inputs = tokenizer(
            email_text(email),
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            probabilities = torch.softmax(model(**inputs).logits[0], dim=-1).cpu().numpy()
        best = int(np.argmax(probabilities))
        rows.append({
            "email": number,
            "assunto": email.get("subject", ""),
            "previsão": LABELS[best],
            "confiança": probabilities[best],
            **{label: probabilities[index] for index, label in enumerate(LABELS)},
        })
    display(pd.DataFrame(rows).style.format(
        {"confiança": "{:.1%}", **{label: "{:.1%}" for label in LABELS}}
    ))

,email,assunto,previsão,confiança,Pedido de Informação,Pedido de Encomenda,SPAM
0,1,Re: Pedido de Cotação,Pedido de Informação,99.9%,99.9%,0.0%,0.0%
